# True / False Experimentation

## Motivation

Anthropic's circuit-tracing work studied how Gemma 2 (2B) answers the open-ended prompt:

> *"Fact: the capital of the state containing Dallas is"*

In that setting the model naturally completes the sentence with a city name, so the token `▁Austin` received ~41 % of the probability mass — a strong, clean signal to trace. Attribution graphs built on the `Austin − Dallas` logit difference were correspondingly rich.

This notebook investigates a **different family of questions**: those that ask the model to judge the truth value of a statement, producing **`True`** or **`False`** as its first token. Empirically, these tokens receive much less probability mass (~5–15 %), and the `True − False` attribution score is correspondingly smaller. Before blaming the model or the methodology, we need to understand **why** — and that is exactly what the three experiments below address.

### Three experiments

| # | Name | What it answers |
|---|------|-----------------|
| 1 | **Baseline softmax** | What is the model's theoretical probability distribution over all tokens for this prompt? |
| 2 | **Empirical sampling** | Does repeated sampling match the softmax prediction? Where do `True` / `False` rank? |
| 3 | **Multi-step rollout** | If the model does not say `True`/`False` at step 1, what does it say — and does a verdict ever appear? |
| 4 | **Attribution graph** | For the `True − False` direction, which internal features drive the model's answer? |

---

**How to change the prompt:** edit the `PROMPT` variable in the **Experiment Configuration** cell below. Every cell downstream uses it automatically.

In [1]:
# One-time dependency bootstrap for a fresh pod/kernel.
# Run this cell once on a new pod, then restart the kernel and run from the top.

# %pip install --no-cache-dir -e /workspace/thesis_circuit_breaker

In [2]:
#@title Runtime + Hugging Face Setup (Local/Runpod) { display-mode: "form" }

import os
import sys
from pathlib import Path

IN_RUNPOD = os.environ.get("RUNPOD_POD_ID") is not None

print("Configuring local/Runpod environment...")

def _find_repo_root() -> Path | None:
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "circuit_tracer" / "__init__.py").is_file():
            return directory

    # On Runpod, repo is often under /workspace with different folder names.
    workspace = Path("/workspace")
    if workspace.is_dir():
        for child in workspace.iterdir():
            if child.is_dir() and (child / "circuit_tracer" / "__init__.py").is_file():
                return child

    # Optional explicit override for custom locations.
    repo_override = os.environ.get("CT_REPO_DIR")
    if repo_override:
        override_path = Path(repo_override).expanduser().resolve()
        if (override_path / "circuit_tracer" / "__init__.py").is_file():
            return override_path

    return None

_root = _find_repo_root()
if _root is not None:
    root_str = str(_root)
    if root_str not in sys.path:
        sys.path.insert(0, root_str)
    print(f"Repo root on sys.path: {_root}")

    # Ensure core project dependencies are present in the active kernel env.
    import importlib.util
    import subprocess

    required_modules = [
        "safetensors",
        "transformers",
        "tokenizers",
        "nnsight",
        "transformer_lens",
    ]
    missing = [m for m in required_modules if importlib.util.find_spec(m) is None]
    if missing:
        raise RuntimeError(
            "Missing dependencies detected: "
            f"{missing}. Install once in this kernel env with: `pip install -e {_root}` "
            "then restart kernel and rerun from the top."
        )

    # Pin HF stack to versions compatible with this repo (see pyproject.toml).
    # Newer transformers releases can remove symbols like BertForPreTraining and break lazy imports.
    from importlib.metadata import version as _pkg_version

    def _parse(v: str) -> tuple[int, int, int]:
        core = v.split("+", 1)[0].strip()
        parts = core.split(".")
        nums: list[int] = []
        for p in parts[:3]:
            digits = "".join(ch for ch in p if ch.isdigit())
            nums.append(int(digits) if digits else 0)
        while len(nums) < 3:
            nums.append(0)
        return nums[0], nums[1], nums[2]

    def _lt(a: str, b: str) -> bool:
        return _parse(a) < _parse(b)

    def _gt(a: str, b: str) -> bool:
        return _parse(a) > _parse(b)

    def _gte(v: str, minimum: str) -> bool:
        return not _lt(v, minimum)

    def _pin_hf_stack() -> None:
        try:
            hf_hub_v = _pkg_version("huggingface_hub")
        except Exception:
            hf_hub_v = ""

        try:
            tf_v = _pkg_version("transformers")
        except Exception:
            tf_v = ""

        needs_pin = False
        if hf_hub_v and _gte(hf_hub_v, "1.0.0"):
            needs_pin = True

        if tf_v and (_lt(tf_v, "4.56.0") or _gt(tf_v, "4.57.3")):
            needs_pin = True

        if needs_pin:
            print("Pinning huggingface_hub + transformers to repo-compatible versions...")
            subprocess.check_call(
                [
                    sys.executable,
                    "-m",
                    "pip",
                    "install",
                    "-q",
                    "--no-cache-dir",
                    "huggingface_hub<1.0.0",
                    "transformers>=4.56.0,<=4.57.3",
                ]
            )
            raise RuntimeError(
                "Pinned huggingface_hub/transformers. Please restart the kernel, then rerun from the top."
            )

    _pin_hf_stack()
else:
    print(
        "Could not locate local circuit_tracer repo. "
        "Clone your repo into /workspace or set CT_REPO_DIR to your repo path."
    )

# Avoid torch import failures after dependency drift in long-lived pod environments.
try:
    from typing_extensions import TypeIs  # type: ignore[attr-defined]
except Exception:
    print("typing_extensions is too old for current torch; upgrading typing_extensions...")
    %pip install -q --no-cache-dir "typing_extensions>=4.12.2"
    raise RuntimeError(
        "Installed typing_extensions>=4.12.2. Please restart the kernel, then rerun from the top."
    )

# Load local .env when available (useful for local runs). On Runpod, env vars are usually set in pod config.
try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

if load_dotenv is not None and _root is not None:
    _env_file = _root / ".env"
    if _env_file.is_file():
        load_dotenv(_env_file)
        print(f"Loaded {_env_file} (e.g. HF_TOKEN for Hugging Face).")

# Standardize token env names so huggingface_hub can pick it up reliably.
hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
if hf_token and not os.environ.get("HUGGINGFACE_HUB_TOKEN"):
    os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token

# Configure persistent cache/output paths when running on Runpod.
if IN_RUNPOD:
    persistent_root = Path(os.environ.get("RUNPOD_PERSISTENT_ROOT", "/workspace")).resolve()
    hf_home = persistent_root / "hf"
    cache_dirs = {
        "HF_HOME": hf_home,
        "HUGGINGFACE_HUB_CACHE": hf_home / "hub",
        "TRANSFORMERS_CACHE": hf_home / "transformers",
        "HF_DATASETS_CACHE": hf_home / "datasets",
        "TORCH_HOME": persistent_root / "torch",
        "CT_OUTPUT_DIR": persistent_root / "results" / "true_false_exp",
    }
    for key, path in cache_dirs.items():
        path.mkdir(parents=True, exist_ok=True)
        os.environ[key] = str(path)

    print("Runpod detected. Persistent storage configured:")
    print(f"  root={persistent_root}")
    for key in [
        "HF_HOME",
        "HUGGINGFACE_HUB_CACHE",
        "TRANSFORMERS_CACHE",
        "HF_DATASETS_CACHE",
        "TORCH_HOME",
        "CT_OUTPUT_DIR",
    ]:
        print(f"  {key}={os.environ[key]}")

from huggingface_hub import get_token

if get_token() is None:
    print(
        "No Hugging Face token in this environment. google/gemma-2-2b is gated: accept the "
        "license at https://huggingface.co/google/gemma-2-2b then set HF_TOKEN in your env."
    )
    print("If on Runpod: set HF_TOKEN in pod env vars and restart kernel.")
else:
    print("Hugging Face token detected; model/cache should be reusable across runs.")

Configuring local/Runpod environment...
Repo root on sys.path: /workspace/thesis_circuit_breaker
Runpod detected. Persistent storage configured:
  root=/workspace
  HF_HOME=/workspace/hf
  HUGGINGFACE_HUB_CACHE=/workspace/hf/hub
  TRANSFORMERS_CACHE=/workspace/hf/transformers
  HF_DATASETS_CACHE=/workspace/hf/datasets
  TORCH_HOME=/workspace/torch
  CT_OUTPUT_DIR=/workspace/results/true_false_exp
Hugging Face token detected; model/cache should be reusable across runs.


In [3]:
# GPU and environment sanity checks — run at the start of every session.
import os
import torch

print(f"torch version: {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"gpu: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU not detected. Runs will be very slow on CPU.")

for key in [
    "HF_HOME",
    "HUGGINGFACE_HUB_CACHE",
    "TRANSFORMERS_CACHE",
    "HF_DATASETS_CACHE",
    "TORCH_HOME",
    "CT_OUTPUT_DIR",
]:
    print(f"{key}={os.environ.get(key, '<unset>')}")

print("HF token set:", bool(os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")))

torch version: 2.6.0+cu124
cuda available: True
gpu: NVIDIA RTX A4500
HF_HOME=/workspace/hf
HUGGINGFACE_HUB_CACHE=/workspace/hf/hub
TRANSFORMERS_CACHE=/workspace/hf/transformers
HF_DATASETS_CACHE=/workspace/hf/datasets
TORCH_HOME=/workspace/torch
CT_OUTPUT_DIR=/workspace/results/true_false_exp
HF token set: True


In [4]:
# All imports — collected here so the rest of the notebook stays clean.

import gc
import os
import sys
import json
from pathlib import Path
from collections import Counter

import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.attribution.targets import CustomTarget
from circuit_tracer.utils import create_graph_files
from circuit_tracer.utils.demo_utils import (
    cleanup_cuda,
    display_token_probs,
    display_topk_token_predictions,
    display_top_features_comparison,
    get_top_features,
)

/workspace/venvs/ct/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


## Experiment Configuration

**This is the only cell you need to edit** to run the experiments on a different prompt or question.

- Change `PROMPT` to any statement-judgement prompt you want to investigate.
- `TOKEN_TRUE` and `TOKEN_FALSE` are the Gemma tokenizer surface forms for the verdict tokens (the leading `▁` represents a space character — important for Gemma's SentencePiece tokenizer).
- `N_SAMPLES` controls how many times we sample in Experiment 2. 100 is a good starting point; increase to 300+ for tighter empirical estimates.
- `N_STEPS` controls how many tokens the rollout generates in Experiment 3.
- `TRANSCODER_LAZY_ENCODER` — defer loading CLT encoder weights from disk until needed (lowers VRAM spikes during `from_pretrained`).
- `CUDA_MIN_FREE_GIB` — heuristic gate before load (~8 GiB+ often works with `TRANSCODER_LAZY_ENCODER` on a 24 GB GPU with only this notebook). Raise if loads still OOM; set `0.0` to disable the check entirely.

In [5]:
# ── EXPERIMENT CONFIGURATION ─────────────────────────────────────────────────
# Edit this cell to change the question. Everything downstream uses these variables.


# Verdict tokens (Gemma SentencePiece: leading ▁ = space before the word)
TOKEN_TRUE  = " True"
TOKEN_FALSE = " False"




# Experiment 4 — attribution
ATTR_KWARGS = dict(
    batch_size=256,
    max_feature_nodes=8192,
    offload="disk",
    verbose=True,
)

# Output slug — graph files will be saved under CT_OUTPUT_DIR / OUTPUT_SLUG
OUTPUT_SLUG = "true_false_exp"

# Smaller CUDA footprint while loading transcoder shards (recommended).
TRANSCODER_LAZY_ENCODER = True

# Before loading Gemma + CLT, require roughly this much free VRAM on device 0 (soft guard).
# 8 GiB often suffices with TRANSCODER_LAZY_ENCODER; raise if load still hits CUDA OOM; 0 = skip check.
CUDA_MIN_FREE_GIB = 8.0


## Model Loading

We use `ReplacementModel` from the `circuit_tracer` library. This is a special wrapper around **Gemma 2 (2B)** that substitutes the model's internal MLP layers with a *cross-layer transcoder* (CLT) — a sparse, more interpretable component trained to approximate the same computation.

Why bother? The original MLP neurons are *polysemantic* — each neuron fires for many unrelated concepts at once, making them hard to interpret. CLT features are *sparse* and typically represent a single, human-readable concept (e.g. "triangle geometry", "mathematical inequality"). Attribution graphs computed on this replacement model tell us which such features contributed most to a specific output.

The model weights are downloaded from Hugging Face on first use and cached in `/workspace/hf` on Runpod, so subsequent runs are fast.

In [6]:
# Drop Python references and stale CUDA cache peaks before allocating Gemma + CLT.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_b, total_b = torch.cuda.mem_get_info()
    free_gib = free_b / (1024**3)
    total_gib = total_b / (1024**3)
    print(f"CUDA device 0: {free_gib:.2f} GiB free of {total_gib:.2f} GiB total (after gc + empty_cache)")

    if CUDA_MIN_FREE_GIB > 0 and free_gib < CUDA_MIN_FREE_GIB:
        raise RuntimeError(
            f"Not enough free GPU memory to load ReplacementModel reliably "
            f"({free_gib:.2f} GiB free, need roughly >= {CUDA_MIN_FREE_GIB} GiB).\n\n"
            "This usually means another process is using most of the GPU. "
            "Run `nvidia-smi`, stop other Jupyter kernels or Python jobs, restart this kernel, "
            "then rerun from the configuration cell onward. "
            "If you intentionally share GPU, temporarily set CUDA_MIN_FREE_GIB = 0.0 in the experiment config."
        )


CUDA device 0: 8.30 GiB free of 19.70 GiB total (after gc + empty_cache)


In [7]:
model_name       = "google/gemma-2-2b"
transcoder_name  = "gemma"
backend          = "transformerlens"  # change to 'nnsight' for the NNSight backend

from circuit_tracer.utils import get_default_device

device = get_default_device()
print(f"Loading model on {device} (cuda > mps > cpu)")
if device.type == "cpu" and sys.platform == "darwin":
    _mps = getattr(torch.backends, "mps", None)
    _built = _mps.is_built() if _mps is not None else False
    _avail = _mps.is_available() if _mps is not None else False
    print(
        f"macOS: torch.backends.mps.is_built()={_built}, is_available()={_avail}. "
        "If both are False, this PyTorch is likely CPU-only."
    )

model = ReplacementModel.from_pretrained(
    model_name,
    transcoder_name,
    dtype=torch.bfloat16,
    backend=backend,
    device=device,
    lazy_encoder=TRANSCODER_LAZY_ENCODER,
    lazy_decoder=True,
)

tokenizer = model.tokenizer
print("Model loaded.")

Loading model on cuda (cuda > mps > cpu)


Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer
Model loaded.


---

## Section 1 — Baseline: What Does the Model Predict?

**Background: softmax and the next-token distribution**

A language model does not produce a single answer — it produces a **probability distribution over the entire vocabulary** (roughly 256,000 tokens for Gemma). This is computed by applying the *softmax* function to a vector of raw scores called *logits*:

$$P(\text{token}_i) = \frac{e^{\text{logit}_i}}{\sum_j e^{\text{logit}_j}}$$

Every token gets a non-zero probability. The model's "answer" is whichever token has the highest probability — but many other tokens also receive significant mass.

**Why True/False are often not at the top**

When asked an open-ended question like *"the capital of Texas is"*, the probability is concentrated on a small set of city names, so `▁Austin` naturally dominates. When asked a judgement question like *"True or False"*, the model must **decide to produce a label** — but:

- `\n` (newline) is often the single most probable token after a question, because many training examples end with a newline.
- The model may have learned to begin an explanation rather than a label (e.g. `▁The`, `▁Yes`).
- Both `▁True` and `▁False` are in the vocabulary, but if instruction-following data is sparse relative to general text, their combined probability may still be small.

This first cell shows the full picture: we run one forward pass, display the top tokens, and highlight where `True` and `False` rank.

In [8]:
prompt1 = "Statement: For any triangle, the sum of any two sides is greater than the third side. Answer:"
prompt2 = "Statement: For any triangle, the sum of any two sides is greater than the third side. Answer with exactly one word: True or False."
prompt3 = "Statement: For any triangle, the sum of any two sides is less than the third side. Answer:"

PROMPT = prompt2

In [9]:
# Resolve vocabulary IDs for the verdict tokens.
idx_true  = tokenizer.encode(TOKEN_TRUE,  add_special_tokens=False)[-1]
idx_false = tokenizer.encode(TOKEN_FALSE, add_special_tokens=False)[-1]
print(f"{TOKEN_TRUE!r}  → vocab id {idx_true}")
print(f"{TOKEN_FALSE!r} → vocab id {idx_false}")

# Single forward pass to get the logit vector at the last token position.
input_ids = model.ensure_tokenized(PROMPT)
with torch.no_grad():
    baseline_logits, _ = model.get_activations(input_ids)

# Show probabilities for our two verdict tokens.
print("\n--- Verdict token probabilities ---")
display_token_probs(
    baseline_logits,
    token_ids=[idx_true, idx_false],
    labels=[TOKEN_TRUE, TOKEN_FALSE],
    title="Baseline: True vs False",
)

# Show top-15 tokens so we can see where True/False rank among all candidates.
probs_all = torch.softmax(baseline_logits.squeeze(0)[-1].float(), dim=-1)
topk = torch.topk(probs_all, k=15)

print("\n--- Top-15 next tokens ---")
print(f"{'Rank':<6} {'Token':<20} {'Probability':>12} {'Logit':>10}")
print("-" * 52)
for rank, (token_id, prob) in enumerate(zip(topk.indices.tolist(), topk.values.tolist()), start=1):
    token_str = tokenizer.decode([token_id])
    logit_val = baseline_logits.squeeze(0)[-1, token_id].item()
    marker = " ◄" if token_id in (idx_true, idx_false) else ""
    print(f"{rank:<6} {repr(token_str):<20} {prob:>11.3%} {logit_val:>10.4f}{marker}")

' True'  → vocab id 5569
' False' → vocab id 7662

--- Verdict token probabilities ---


Token,Probability,Logit
True,1.427%,23.8750
False,0.219%,22.0000



--- Top-15 next tokens ---
Rank   Token                 Probability      Logit
----------------------------------------------------
1      '\n\n'                   41.695%    27.2500
2      '\n'                     11.946%    26.0000
3      ' If'                     4.980%    25.1250
4      ' Explain'                4.980%    25.1250
5      ' Jus'                    3.423%    24.7500
6      '\n\n\n'                  1.617%    24.0000
7      ' ('                      1.427%    23.8750
8      ' '                       1.427%    23.8750
9      ' The'                    1.427%    23.8750
10     ' True'                   1.427%    23.8750 ◄
11     ' Why'                    1.259%    23.7500
12     ' Explanation'            0.981%    23.5000
13     ' Statement'              0.981%    23.5000
14     ' Please'                 0.981%    23.5000
15     ' A'                      0.981%    23.5000


---

## Section 2 — Empirical First-Token Distribution

**What are we testing?**

The softmax probabilities above are the model's *theoretical* prediction for the first token. But are they real? One way to verify: **sample the first token repeatedly** and compare the observed frequencies to the predicted probabilities.

If the model assigns `▁True` a 10% probability, we expect it to appear roughly 10 times in 100 independent draws — not exactly 10, because sampling is random, but close.

**How we sample**

We use `torch.multinomial` with **temperature = 1.0**. At this temperature, the sampling distribution is exactly the softmax — no tokens are boosted or suppressed. Each call returns one token drawn proportionally to its probability.

We repeat this `N_SAMPLES` times (default: 100) and count how often each token appeared. This gives us an **empirical distribution** to compare against the theoretical one.

**What to expect**

- For tokens with probability > 10%, the empirical count should be reasonably close to theoretical (within a few percent).
- For tokens with probability < 2%, 100 samples is too few to estimate reliably — we might see 0 counts for tokens that theoretically have non-zero probability.
- If ` True` and ` False` appear rarely in the sample, this is entirely consistent with the small probabilities shown in Section 1 — it is not a bug or a model failure.

In [10]:
# Experiment 2 — empirical sampling
N_SAMPLES   = 1000     # number of independent first-token samples
SAMPLE_TEMP = 1.0     # temperature=1.0 samples exactly from the softmax distribution
SEED        = 42      # for reproducibility


In [11]:
torch.manual_seed(SEED)

# Compute the sampling distribution (temperature scaling, then softmax).
raw_logits_last = baseline_logits.squeeze(0)[-1].float()
scaled_logits   = raw_logits_last / SAMPLE_TEMP  # at temp=1.0 this is a no-op
probs_sample    = torch.softmax(scaled_logits, dim=-1)

# Draw N_SAMPLES tokens from this distribution (with replacement — each draw is independent).
samples = torch.multinomial(probs_sample, num_samples=N_SAMPLES, replacement=True)
counts  = Counter(samples.tolist())

# Convert to a sorted table: (token_str, empirical_count, theoretical_probability)
rows = []
for token_id, count in counts.most_common():
    token_str  = tokenizer.decode([token_id])
    theory_pct = probs_sample[token_id].item() * 100
    empirical_pct = count / N_SAMPLES * 100
    rows.append((token_str, count, empirical_pct, theory_pct))

# Always include the verdict tokens even if they were never sampled.
sampled_ids = set(counts.keys())
for vid, vstr in [(idx_true, TOKEN_TRUE), (idx_false, TOKEN_FALSE)]:
    if vid not in sampled_ids:
        theory_pct = probs_sample[vid].item() * 100
        rows.append((vstr, 0, 0.0, theory_pct))

# Print the first 15 rows only.
print(f"Empirical first-token distribution over {N_SAMPLES} samples (temp={SAMPLE_TEMP}, seed={SEED})")
print("First 15 rows:")
print(f"{'Token':<22} {'Count':>6} {'Empirical %':>12} {'Softmax %':>12} {'Diff':>8}")
print("-" * 64)
for token_str, count, emp_pct, th_pct in sorted(rows, key=lambda r: -r[1])[:20]:
    diff = emp_pct - th_pct
    marker = " ◄" if token_str.strip() in ("True", "False") else ""
    print(f"{repr(token_str):<22} {count:>6} {emp_pct:>11.1f}% {th_pct:>11.3f}% {diff:>+7.1f}%{marker}")

Empirical first-token distribution over 1000 samples (temp=1.0, seed=42)
First 15 rows:
Token                   Count  Empirical %    Softmax %     Diff
----------------------------------------------------------------
'\n\n'                    436        43.6%      41.695%    +1.9%
'\n'                      127        12.7%      11.946%    +0.8%
' Explain'                 50         5.0%       4.980%    +0.0%
' If'                      37         3.7%       4.980%    -1.3%
' Jus'                     31         3.1%       3.423%    -0.3%
' ('                       19         1.9%       1.427%    +0.5%
' '                        18         1.8%       1.427%    +0.4%
' The'                     16         1.6%       1.427%    +0.2%
'\n\n\n'                   14         1.4%       1.617%    -0.2%
' Explanation'             13         1.3%       0.981%    +0.3%
' In'                      13         1.3%       0.674%    +0.6%
' True'                    11         1.1%       1.427%    -0.3% ◄


---

## Section 3 — Multi-Step Rollout: Where Does the Model "Answer"?

**The question**

Section 2 showed that ` True` and ` False` may not be the most probable *first* token. But maybe the model still gives a correct verdict — just not at position 1. Perhaps it starts with a newline, then `True`. Or perhaps it decides to write a full-sentence explanation first and only labels the answer at the end.

This section investigates by **auto-regressively generating `N_STEPS` tokens** after the prompt, one at a time, and recording the sequence.

Implementation: the code mirrors TransformerLens **`HookedTransformer.generate`** (same **`past_kv_cache`**, greedy argmax each step), but runs a small local loop so we can keep per-step outputs that **`generate`** discards.

**KV caching** recomputes only the *new* token positions each step. A naive loop that reran **`model(full_prefix)`** every time rebuilt logits for **every** position — the unembed produces a **`[batch, seq, vocab]`** tensor each pass and dominated VRAM when headroom was tight.

Some stacks report **mixed BF16/FP32 tensors inside grouped-query attention** when KV caching is enabled; the rollout cell installs a scoped **attention-score matmul in float32** patch so KV rollout stays workable.

Each step prints the **softmax probability of the greedy token** (the probability mass at the argmax). If rollout still hits CUDA **OOM**, free the GPU elsewhere first; lower **`N_STEPS`** if peak memory is insufficient.

**Drivers / containers:** **`nvidia-smi` can report ~full GPU memory yet list no processes.** PyTorch errors may still mention multiple PIDs. Treat “almost no free CUDA memory” as “this session cannot allocate another Gemma-forward peak” until you restart the kernel or pod.

**Greedy decoding**

We use **greedy decoding**: at each step we pick the single most probable next token (argmax of the logits). This is deterministic and easy to trace — the same prompt always produces the same sequence.

Greedy decoding is different from sampling (Section 2): here we always take the *most likely* token, not a randomly drawn one.

**What to look for**

- Does the model produce ` True` or ` False` at all within 15 steps?
- If yes, at what position? And what came before it?
- If no, what kind of continuation does the model generate? (Explanation? Repetition? Formatting?)

The answers inform what attribution target makes sense: if the verdict is at position 3, attributing at position 3 is more meaningful than at position 1.

In [12]:
N_STEPS = 25          # how many tokens to generate after the prompt

In [13]:
print(f"Greedy rollout: generating up to {N_STEPS} new tokens after the prompt.")
print(f"Verdict tokens: {TOKEN_TRUE!r} (id={idx_true}), {TOKEN_FALSE!r} (id={idx_false})")
print("=" * 60)

if torch.cuda.is_available():
    free_gib = torch.cuda.mem_get_info()[0] / (1024**3)
    total_gib = torch.cuda.mem_get_info()[1] / (1024**3)
    print(f"CUDA free before rollout: {free_gib:.2f} / {total_gib:.2f} GiB")
    if free_gib < 3.0:
        print(
            "\n*** WARNING: very little free VRAM. ***\n"
            "PyTorch may still report almost full GPU usage even when `nvidia-smi` hides PIDs "
            "(container/driver views differ). Garbage collection cannot free VRAM owned elsewhere.\n"
            "Restart this kernel alone on the GPU, restart the pod, or set "
            "`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`.\n"
        )

# Same tokenization path as Sections 1–2 (dummy BOS special token handled inside ensure_tokenized).
prompt_batch = model.ensure_tokenized(PROMPT)
if prompt_batch.ndim == 1:
    prompt_batch = prompt_batch.unsqueeze(0)

cleanup_cuda()


def _patch_transformerlens_kv_attn_scores_float32_matmul():
    """Work around mixed BF16/FP32 q vs k inside GQA score matmul when `past_kv_cache` is on.

    Without this, TransformerLens may raise ``expected scalar type Float but found BFloat16``.
    Falling back to ``use_past_kv_cache=False`` replays full context each step and can OOM.
    """
    from transformer_lens.components import abstract_attention as _tl_aa

    cls = _tl_aa.AbstractAttention
    if getattr(cls.calculate_attention_scores, "_ct_kv_bf16_gqa_patch", False):
        return
    _orig = cls.calculate_attention_scores

    def _wrapped(self, q, k):
        return _orig(self, q.float(), k.float()).to(q.dtype)

    _wrapped._ct_kv_bf16_gqa_patch = True  # type: ignore[attr-defined]
    cls.calculate_attention_scores = _wrapped  # type: ignore[method-assign]


_patch_transformerlens_kv_attn_scores_float32_matmul()


def _tl_get_device_for_block_index(index: int, cfg, device=None):
    """TL 3.x removed ``utilities.devices.get_device_for_block_index``; TL 2.x still has it.

    Mirrors TransformerLens semantics for splitting blocks across GPUs (most pods: ``n_devices`` 1).
    """
    import torch

    try:
        from transformer_lens.utilities import devices as _tl_dm

        _fn = getattr(_tl_dm, "get_device_for_block_index", None)
        if _fn is not None:
            return _fn(index, cfg, device)
    except Exception:
        pass

    assert cfg.device is not None
    n_dev = getattr(cfg, "n_devices", 1) or 1
    layers_per_device = cfg.n_layers // n_dev
    if device is None:
        device = cfg.device
    device = torch.device(device)
    if device.type == "cpu":
        return device
    device_index = (device.index or 0) + (
        index // layers_per_device if layers_per_device else 0
    )
    return torch.device(device.type, device_index)


_CT_MIN_HOOKED_KV_CACHE_CLS = None


def _minimal_hooked_kv_cache_class():
    """Vendored KV cache (same behaviour as ``transformer_lens.past_key_value_caching``).

    Some containers ship a broken/incomplete ``transformer_lens`` tree without
    ``past_key_value_caching.py``. This keeps the rollout cell self-contained.
    """
    global _CT_MIN_HOOKED_KV_CACHE_CLS
    if _CT_MIN_HOOKED_KV_CACHE_CLS is not None:
        return _CT_MIN_HOOKED_KV_CACHE_CLS

    from dataclasses import dataclass
    from typing import List

    import torch

    @dataclass
    class _KVEntry:
        past_keys: torch.Tensor
        past_values: torch.Tensor
        frozen: bool = False

        @classmethod
        def init_cache_entry(cls, cfg, device, batch_size: int = 1):
            n_heads = cfg.n_key_value_heads if cfg.n_key_value_heads is not None else cfg.n_heads
            return cls(
                past_keys=torch.empty(
                    (batch_size, 0, n_heads, cfg.d_head), device=device, dtype=cfg.dtype
                ),
                past_values=torch.empty(
                    (batch_size, 0, n_heads, cfg.d_head), device=device, dtype=cfg.dtype
                ),
            )

        def append(self, new_keys, new_values):
            updated_keys = torch.cat([self.past_keys, new_keys], dim=1)
            updated_values = torch.cat([self.past_values, new_values], dim=1)
            if not self.frozen:
                self.past_keys = updated_keys
                self.past_values = updated_values
            return updated_keys, updated_values

    @dataclass
    class _KVCache:
        entries: List[_KVEntry]
        previous_attention_mask: torch.Tensor
        frozen: bool = False

        @classmethod
        def init_cache(cls, cfg, device, batch_size: int = 1):
            return cls(
                entries=[
                    _KVEntry.init_cache_entry(
                        cfg,
                        _tl_get_device_for_block_index(i, cfg, device),
                        batch_size,
                    )
                    for i in range(cfg.n_layers)
                ],
                previous_attention_mask=torch.empty(
                    (batch_size, 0),
                    device=device,
                    dtype=torch.int,
                ),
            )

        def freeze(self):
            self.frozen = True
            for e in self.entries:
                e.frozen = True

        def unfreeze(self):
            self.frozen = False
            for e in self.entries:
                e.frozen = False

        def append_attention_mask(self, attention_mask):
            attention_mask = attention_mask.to(self.previous_attention_mask.device)
            updated_mask = torch.cat([self.previous_attention_mask, attention_mask], dim=-1)
            if not self.frozen:
                self.previous_attention_mask = updated_mask
            return updated_mask

        def __getitem__(self, idx):
            return self.entries[idx]

    _CT_MIN_HOOKED_KV_CACHE_CLS = _KVCache
    return _KVCache


def _tl_hooked_kv_cache_cls():
    """Resolve ``HookedTransformerKeyValueCache`` across TransformerLens installs.

    Prefer the public package export; some environments omit ``past_key_value_caching`` from
    subpackage imports while the file still exists on disk.
    """
    import importlib
    import importlib.util
    import pathlib

    try:
        from transformer_lens import HookedTransformerKeyValueCache as _KV

        return _KV
    except ImportError:
        pass

    try:
        return importlib.import_module(
            "transformer_lens.past_key_value_caching"
        ).HookedTransformerKeyValueCache
    except (ModuleNotFoundError, AttributeError):
        pass

    import transformer_lens as tl

    root = pathlib.Path(tl.__file__).resolve().parent
    path = root / "past_key_value_caching.py"
    if path.is_file():
        spec = importlib.util.spec_from_file_location(
            "transformer_lens.past_key_value_caching",
            path,
        )
        if spec is None or spec.loader is None:
            raise ImportError("Could not load past_key_value_caching spec.") from None
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        return mod.HookedTransformerKeyValueCache

    # Last resort: package on disk omitted past_key_value_caching.py entirely.
    return _minimal_hooked_kv_cache_class()


# ``HookedTransformer.generate`` drops per-step logits. Mirror its KV greedy loop to record
# softmax probability of each greedy token (same forward path, same VRAM profile as generate).


def _greedy_rollout_tokens_and_probs_kv(
    tl_model,
    input_tokens_cpu: torch.Tensor,
    max_new_tokens: int,
) -> tuple[torch.Tensor, list[float]]:
    from transformer_lens import utils as tl_utils

    HookedTransformerKeyValueCache = _tl_hooked_kv_cache_cls()

    prepend_bos = tl_utils.USE_DEFAULT_VALUE
    padding_side = tl_utils.USE_DEFAULT_VALUE

    with tl_utils.LocallyOverridenDefaults(
        tl_model, prepend_bos=prepend_bos, padding_side=padding_side
    ):
        batch_size = input_tokens_cpu.shape[0]
        device = _tl_get_device_for_block_index(0, tl_model.cfg)
        input_on_dev = input_tokens_cpu.to(device)
        past_kv_cache = HookedTransformerKeyValueCache.init_cache(
            tl_model.cfg, tl_model.cfg.device, batch_size
        )
        embeds = tl_model.embed(input_on_dev)
        tl_model.eval()
        sampled_tokens_list: list[torch.Tensor] = []
        step_probs: list[float] = []
        start_at_layer = 0

        for index in range(max_new_tokens):
            pos_offset = tl_model.get_pos_offset(past_kv_cache, batch_size)
            tokens = torch.zeros((embeds.size(0), embeds.size(1)), dtype=torch.int)
            attention_mask = tl_utils.get_attention_mask(
                tl_model.tokenizer,
                tokens,
                False if prepend_bos is None else prepend_bos,
            ).to(device)
            residual, shortformer_pe = tl_model.get_residual(
                embeds,
                pos_offset,
                return_shortformer_pos_embed=True,
                device=device,
                attention_mask=attention_mask,
            )
            if index > 0:
                logits = tl_model.forward(
                    residual[:, -1:],
                    return_type="logits",
                    prepend_bos=prepend_bos,
                    padding_side=padding_side,
                    past_kv_cache=past_kv_cache,
                    start_at_layer=start_at_layer,
                    shortformer_pos_embed=shortformer_pe,
                )
            else:
                logits = tl_model.forward(
                    residual,
                    return_type="logits",
                    prepend_bos=prepend_bos,
                    padding_side=padding_side,
                    past_kv_cache=past_kv_cache,
                    start_at_layer=start_at_layer,
                    shortformer_pos_embed=shortformer_pe,
                )
            final_logits = logits[:, -1, :].float()
            probs = torch.softmax(final_logits, dim=-1)
            sampled = final_logits.argmax(-1).to(
                _tl_get_device_for_block_index(0, tl_model.cfg)
            )
            step_probs.append(probs[0, sampled[0]].item())
            sampled_tokens_list.append(sampled.unsqueeze(1))
            embeds = torch.hstack(
                [embeds, tl_model.embed(sampled.unsqueeze(-1))]
            )

        new_tok = torch.cat(sampled_tokens_list, dim=1)
        full = torch.cat((input_on_dev, new_tok), dim=1)
        return full, step_probs


with torch.inference_mode():
    full_ids, greedy_step_probs = _greedy_rollout_tokens_and_probs_kv(
        model, prompt_batch, N_STEPS
    )

cleanup_cuda()

prefix_len = prompt_batch.shape[1]
new_ids_t = full_ids[0, prefix_len:]
generated: list[tuple[int, int, str, float]] = []
for step, (next_id, prob) in enumerate(
    zip(new_ids_t.tolist(), greedy_step_probs, strict=True),
    start=1,
):
    next_str = tokenizer.decode([next_id])
    generated.append((step, next_id, next_str, prob))

# Display the rollout table.
print(f"{'Step':<6} {'Token':<22} {'Prob':>8}  Note")
print("-" * 55)
for step, token_id, token_str, prob in generated:
    note = ""
    if token_id == idx_true:
        note = "  ◄◄ VERDICT: True"
    elif token_id == idx_false:
        note = "  ◄◄ VERDICT: False"
    prob_cell = f"{prob:>7.3%}"
    print(f"{step:<6} {repr(token_str):<22} {prob_cell}{note}")

# Reconstruct and print the full continuation.
continuation = "".join(t for _, _, t, _ in generated)
print("\n--- Full continuation ---")
print(repr(continuation))

# Summary: did a verdict token appear?
verdict_steps = [
    (step, token_str)
    for step, token_id, token_str, _ in generated
    if token_id in (idx_true, idx_false)
]
print("\n--- Verdict summary ---")
if verdict_steps:
    for step, token_str in verdict_steps:
        print(f"Verdict token {token_str!r} appeared at step {step}.")
else:
    print(f"No verdict token appeared within {N_STEPS} steps.")

Greedy rollout: generating up to 25 new tokens after the prompt.
Verdict tokens: ' True' (id=5569), ' False' (id=7662)
CUDA free before rollout: 0.37 / 19.70 GiB

*** WARNING: very little free VRAM. ***
PyTorch may still report almost full GPU usage even when `nvidia-smi` hides PIDs (container/driver views differ). Garbage collection cannot free VRAM owned elsewhere.
Restart this kernel alone on the GPU, restart the pod, or set `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`.



/tmp/ipykernel_15933/2526134376.py:223: DeprecationWarning: The 'utils' module has been deprecated. Please use 'transformer_lens.utilities' instead. Importing from utils.py will be removed in TransformerLens 4.0.
  from transformer_lens import utils as tl_utils


Step   Token                      Prob  Note
-------------------------------------------------------
1      '\n\n'                 44.812%
2      'Step'                 40.882%
3      ' '                    99.950%
4      '1'                    99.966%
5      '\n'                   99.108%
6      '1'                    99.958%
7      ' of'                  99.997%
8      ' '                    99.996%
9      '2'                    58.320%
10     '\n\n'                 95.775%
11     'The'                  22.473%
12     ' statement'           50.195%
13     ' is'                  86.155%
14     ' $\\'                 35.703%
15     'text'                 44.873%
16     '{\\'                  98.565%
17     'textcolor'            49.468%
18     '{#'                   99.773%
19     'c'                    82.056%
20     '3'                    99.990%
21     '4'                    99.998%
22     '6'                    99.997%
23     '3'                    99.999%
24     '2'               

---

## Section 4 — Attribution Graph Analysis

**What is an attribution graph?**

An attribution graph is a directed graph that describes, for a specific prompt and a specific output direction, **which internal features of the model were responsible for producing that output** — and how they influenced each other.

- **Nodes** represent: active CLT features (interpretable sparse components), input token embeddings, reconstruction error terms, and output logits.
- **Edges** represent direct linear effects: how strongly does feature A influence feature B, or feature A influence the final logit?

The graph is built by running the *local replacement model* on the prompt, freezing attention patterns and normalisation denominators, and then computing partial derivatives (attributions) of each output logit with respect to each upstream node.

**Why the True − False contrast?**

We attribute toward the **difference direction** `logit(True) − logit(False)`. This is more informative than attributing to either token alone:

- Features that promote `True` *and* `False` equally will have near-zero attribution to this direction — they are not driving the verdict either way.
- Only features that genuinely tip the balance toward one verdict over the other will appear with high attribution.

In practice, because both `True` and `False` receive small probability mass, the absolute attribution scores will be small compared to the capital-city case. This is expected — a small signal does not mean there is no circuit, it means the circuit's effect is diluted across a broad softmax distribution.

**Interpreting the results**

After running attribution, we:
1. Save the graph files to disk so they can be opened in the `circuit-tracer` local server or on Neuronpedia.
2. Display the top-10 features ranked by total multi-hop influence on the `True − False` direction, with clickable Neuronpedia links.

In [ ]:
# Experiment 4 — attribution
ATTR_KWARGS = dict(
    batch_size=256,
    max_feature_nodes=8192,
    offload="disk",
    verbose=True,
)

In [15]:
# Run attribution toward True and False simultaneously.
# Passing both tokens as string targets lets the library compute the contrast direction internally.
print(f"Running attribution for: {TOKEN_TRUE!r} and {TOKEN_FALSE!r}")
print(f"Prompt:", PROMPT)
print()

graph_tf = attribute(
    prompt=PROMPT,
    model=model,
    attribution_targets=[TOKEN_TRUE, TOKEN_FALSE],
    **ATTR_KWARGS,
)

n_features = graph_tf.active_features.shape[0]
n_targets  = len(graph_tf.logit_targets)
print(f"\nAttribution complete: {n_targets} targets, {n_features} active features.")

# Show the logit probabilities assigned to each target.
print("\nTarget probabilities used for attribution weighting:")
for i, (target, prob) in enumerate(zip(graph_tf.logit_targets, graph_tf.logit_probabilities.tolist())):
    print(f"  [{i}] {target.token_str!r:30s}  prob = {prob:.4f}")

Phase 0: Precomputing activations and vectors


Running attribution for: ' True' and ' False'
Prompt: Statement: For any triangle, the sum of any two sides is greater than the third side. Answer with exactly one word: True or False.



Precomputation completed in 16.51s
Found 32882 active features
Phase 1: Running forward pass
Forward pass completed in 0.56s
Phase 2: Building input vectors
Using 2 specified logit targets with cumulative probability 0.0156
Will include 8192 of 32882 feature nodes
Input vectors built in 5.50s
Phase 3: Computing logit attributions
Logit attributions completed in 0.80s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 8192/8192 [00:13<00:00, 617.80it/s]
Feature attributions completed in 13.26s
Attribution completed in 40.03s



Attribution complete: 2 targets, 32882 active features.

Target probabilities used for attribution weighting:
  [0] ' True'                         prob = 0.0135
  [1] ' False'                        prob = 0.0021


In [16]:
# Save graph files so they can be visualised in the local circuit-tracer server.
#
# After this cell runs, start the server from a terminal:
#   circuit-tracer server --graph_file_dir <output_dir>
# then open the printed URL in your browser.

output_dir = Path(os.environ.get("CT_OUTPUT_DIR", "results")) / OUTPUT_SLUG
output_dir.mkdir(parents=True, exist_ok=True)

create_graph_files(
    graph_or_path=graph_tf,
    slug=OUTPUT_SLUG,
    output_path=str(output_dir),
    node_threshold=0.8,
    edge_threshold=0.98,
)

print(f"Graph files saved to: {output_dir}")
print(f"To visualise, run:")
print(f"  circuit-tracer server --graph_file_dir {output_dir}")

Graph files saved to: /workspace/results/true_false_exp/true_false_exp
To visualise, run:
  circuit-tracer server --graph_file_dir /workspace/results/true_false_exp/true_false_exp


In [17]:
# Display the top-10 most influential features for the True/False attribution.
#
# Each feature is identified by (layer, token_position, feature_index).
# The Neuronpedia links open an interactive dashboard showing which texts activated that feature,
# helping you interpret what concept the feature represents.

features_tf, scores_tf = get_top_features(graph_tf, n=10)

print(f"Top-10 features by multi-hop influence on {TOKEN_TRUE!r} / {TOKEN_FALSE!r}:")
print(f"{'#':<4} {'(layer, pos, feat_idx)':<30} {'Score':>10}  Neuronpedia")
print("-" * 80)
np_model = "gemma-2-2b"
np_set   = "gemmascope-transcoder-16k"
for i, ((layer, pos, feat_idx), score) in enumerate(zip(features_tf, scores_tf), start=1):
    np_url = f"https://www.neuronpedia.org/{np_model}/{layer}-{np_set}/{feat_idx}"
    print(f"{i:<4} ({layer:>2}, {pos:>3}, {feat_idx:>6})          {score:>10.5f}  {np_url}")

# Also render the styled HTML table (clickable links).
display_top_features_comparison(
    feature_sets={f"{TOKEN_TRUE} / {TOKEN_FALSE}": features_tf},
    scores_sets={f"{TOKEN_TRUE} / {TOKEN_FALSE}": scores_tf},
    neuronpedia_model=np_model,
    neuronpedia_set=np_set,
)

cleanup_cuda()

Top-10 features by multi-hop influence on ' True' / ' False':
#    (layer, pos, feat_idx)              Score  Neuronpedia
--------------------------------------------------------------------------------
1    (24,  29,   4640)             0.00023  https://www.neuronpedia.org/gemma-2-2b/24-gemmascope-transcoder-16k/4640
2    ( 0,   2,  16200)             0.00009  https://www.neuronpedia.org/gemma-2-2b/0-gemmascope-transcoder-16k/16200
3    (18,  26,   8253)             0.00009  https://www.neuronpedia.org/gemma-2-2b/18-gemmascope-transcoder-16k/8253
4    (22,  29,   5015)             0.00009  https://www.neuronpedia.org/gemma-2-2b/22-gemmascope-transcoder-16k/5015
5    (23,  29,    816)             0.00008  https://www.neuronpedia.org/gemma-2-2b/23-gemmascope-transcoder-16k/816
6    (24,  29,   1764)             0.00007  https://www.neuronpedia.org/gemma-2-2b/24-gemmascope-transcoder-16k/1764
7    (17,  26,  11405)             0.00007  https://www.neuronpedia.org/gemma-2-2b/17-gemmascope

#,Node,Score
1,"(24, 29, 4640)",0.0002
2,"(0, 2, 16200)",0.0001
3,"(18, 26, 8253)",0.0001
4,"(22, 29, 5015)",0.0001
5,"(23, 29, 816)",0.0001
6,"(24, 29, 1764)",0.0001
7,"(17, 26, 11405)",0.0001
8,"(0, 29, 6051)",0.0001
9,"(20, 29, 9550)",0.0001
10,"(21, 29, 10354)",0.0001


In [18]:
# Tokenized PROMPT — maps attribution table `pos` to tokenizer pieces.
# Uses the same path as Sections 1–3 (`ensure_tokenized`), matching graph positions.

_prompt_row = model.ensure_tokenized(PROMPT)
_prompt_ids = _prompt_row.squeeze(0) if _prompt_row.ndim == 2 else _prompt_row

print("Tokenized PROMPT — pos → token_id → decoded")
print(repr(PROMPT[:120]) + ("..." if len(PROMPT) > 120 else ""))
print("=" * 64)
print(f"{'pos':>4}  {'token_id':>10}   decoded")
print("-" * 64)
for pos, tid in enumerate(_prompt_ids.tolist()):
    print(f"{pos:>4}  {tid:>10}   {repr(tokenizer.decode([tid]))}")
print("=" * 64)
print(f"Total tokens: {len(_prompt_ids)}  (prompt positions run 0 .. {len(_prompt_ids) - 1}).")

Tokenized PROMPT — pos → token_id → decoded
'Statement: For any triangle, the sum of any two sides is greater than the third side. Answer with exactly one word: True'...
 pos    token_id   decoded
----------------------------------------------------------------
   0           2   '<bos>'
   1       12195   'Statement'
   2      235292   ':'
   3        1699   ' For'
   4        1089   ' any'
   5       23022   ' triangle'
   6      235269   ','
   7         573   ' the'
   8        2707   ' sum'
   9         576   ' of'
  10        1089   ' any'
  11        1378   ' two'
  12       10012   ' sides'
  13         603   ' is'
  14        6561   ' greater'
  15        1178   ' than'
  16         573   ' the'
  17        4906   ' third'
  18        2857   ' side'
  19      235265   '.'
  20       10358   ' Answer'
  21         675   ' with'
  22        7880   ' exactly'
  23         974   ' one'
  24        2204   ' word'
  25      235292   ':'
  26        5569   ' True'
  27         689   

## Reading the attribution graph with Neuronpedia and token positions

The **attribution graph** you saved is a compact picture of *which learned features* in the model’s residual stream tend to push the next-token distribution toward “True” versus “False” for this prompt. Nodes are typically **(layer, position, feature index)** triples; directed edges summarize how influence flows between them under the chosen attribution method. The graph is *not* a causal proof of a single mechanism—it is a **hypothesis map**: it tells you where to look when you open the same features in a dedicated viewer or in [Neuronpedia](https://www.neuronpedia.org/).

**Neuronpedia** catalogs **sparse interpretability dictionaries** from several training setups—not only **sparse autoencoders (SAEs)**, which typically **reconstruct** the same activation they encode, but also **sparse transcoders** (e.g. Gemma Scope’s `gemmascope-transcoder-16k`), which approximate **MLP input → MLP output** via a sparse latent rather than vanilla autoencoding. **The links in this notebook use those transcoder dictionaries.** Regardless of variant, each feature id may have a short **label** and **maximum-activation examples**—roughly, “this direction fires on text that looks like …”. That is a **semantic prior**, not a guarantee on every prompt.

The integer **`pos`** in the attribution table is **not** a character offset in the string; it is the **index of the token slot** in the prompt sequence, aligned with how the tokenizer split the text. Run the preceding cell’s printout alongside the feature table: **`pos`** → **`decoded`** lets you anchor each row to **which piece of surface text** the feature attends to or writes through at that depth (early tokens like “Statement … :”, the geometric claim, “Answer … :”, or the final **`True` / `False`** region). Patterns often cluster on **delimiter tokens** (`:`, commas, final `.`) and on the short **instruction tail** (“True or False.”) because those are exactly where format and decision semantics overlap in many instruction-tuned models.

**Attribution scores are on an arbitrary relative scale.** For the **True − False logit contrast**, magnitudes ~1e⁻⁴ simply mean this prompt is **logit-flat** near the greedy argmax—the model marginally favors one label, unlike toy geography facts that produce huge, eye-catching contrasts. Small scores do **not** automatically mean “no interesting circuit”; they mean each listed feature **nudges** the contrast a little. The graph and Neuronpedia together answer: *where* those nudges concentrate in depth and in the prompt, and *what kind of feature* they are in feature space—so you can decide what to visualize or ablate next.

## Interpretation for *this* prompt and run

The configured prompt is the **triangle inequality** statement with an explicit tail: *“Answer with exactly one word: True or False.”* In the attribution run saved in this notebook, ` True` had about **1.4%** next-token probability and ` False` about **0.2%** — so the model is only weakly committed to either label at the position we attribute. That already explains why the **multi-hop influence scores** land in the **`5e-5`–`2.3e-4`** range: each listed feature only **tilts** `logit(True) − logit(False)` slightly; there is no enormous logit gap to decompose.

Below is a reading of the **top-10 `(layer, pos, feat)`** rows **together with the token map** from the preceding cell. Cross-check `pos` → decoded token (e.g. **`2` → `':'`**, **`26` → ` True`**, **`29` → `.`** at the end of “False.”).

| Rank | Node | Score | Neuronpedia (short gloss from the dashboard) | How it fits *this* prompt |
|------|------|-------|-----------------------------------------------|---------------------------|
| 1 | (24, **29**, **4640**) | **0.00023** | [4640](https://www.neuronpedia.org/gemma-2-2b/24-gemmascope-transcoder-16k/4640) — *assertions of truth/falsity*, *logical statements* | Very late layer at the **final period**; aligns with “verdict / Q&A” endings where the model gathers a **binary** readout. |
| 2 | (**0**, **2**, **16200**) | 0.00009 | [16200](https://www.neuronpedia.org/gemma-2-2b/0-gemmascope-transcoder-16k/16200) — *syntax*, *`:`* | **Earliest** residual position on the colon after `Statement`; captures **template / field** structure before any deep geometry. |
| 3 | (18, **26**, **8253**) | 0.00009 | [8253](https://www.neuronpedia.org/gemma-2-2b/18-gemmascope-transcoder-16k/8253) — *“True” (and sometimes “False”) in math / logic* | Sits on the **` True`** token — a direct **lexical–instruction** channel for the constrained answer. |
| 4 | (22, **29**, **5015**) | 0.00009 | [5015](https://www.neuronpedia.org/gemma-2-2b/22-gemmascope-transcoder-16k/5015) — *equations / calculations*, *question syntax* | End of the prompt; mixes **math-reading** with **question framing** — consistent with a geometry claim plus a **forced** question format. |
| 5 | (23, **29**, **816**) | 0.00008 | [816](https://www.neuronpedia.org/gemma-2-2b/23-gemmascope-transcoder-16k/816) — **period** / sentence closure | Same **closing `.`** as rank 1–4; reinforces that many salient features hug the **instruction boundary**, not only the middle of the theorem text. |
| 6 | (24, **29**, **1764**) | 0.00007 | [1764](https://www.neuronpedia.org/gemma-2-2b/24-gemmascope-transcoder-16k/1764) — *math expressions*, *action / imperatives* | Again at the **final token**; ties **“do X”** discourse to the **True/False** choice the user demanded. |
| 7 | (17, **26**, **11405**) | 0.00007 | [11405](https://www.neuronpedia.org/gemma-2-2b/17-gemmascope-transcoder-16k/11405) — *true/false in algorithms / logic* | Parallel to feature 8253 on **` True`**, but slightly earlier in depth — a **logic-predicate** motif on the disjunction site. |
| 8 | (**0**, **29**, **6051**) | 0.00007 | [6051](https://www.neuronpedia.org/gemma-2-2b/0-gemmascope-transcoder-16k/6051) — *periods*, spacing | **Shallow** geometry on the closing punctuation; often such features are **generic format** detectors that still matter for **where** the model “commits” on the next token. |
| 9 | (20, **29**, **9550**) | 0.00006 | [9550](https://www.neuronpedia.org/gemma-2-2b/20-gemmascope-transcoder-16k/9550) — *symbolic math*, *punctuation* | Another late-prompt blend of **stem content** (inequality language) and **delimiters** feeding the contrast. |
| 10 | (21, **29**, **10354**) | 0.00005 | [10354](https://www.neuronpedia.org/gemma-2-2b/21-gemmascope-transcoder-16k/10354) — *explaining*, *statements vs. hypothesis* | Suggests residual **argumentation / justification** structure is being recruited even though the user only asked for **one word** — a useful sanity check that the model may still be “reasoning aloud” internally. |

**Takeaway.** For this specific prompt, the attribution is **not** dominated by a single mid-theorem feature; it is spread across (i) **format** tokens (`:` early, `.` late), (ii) explicit **`True`/logic** positions, and (iii) **math- and explanation-flavored** features anchored on the **instruction tail**. Because the **True − False** signal is **numerically small**, treat the table as **relative ranking** of nudges — and use the Neuronpedia pages to eyeball **maximum-activating examples** and decide which features merit ablation or steering in a follow-up experiment.

*If you change `PROMPT` or rerun attribution, this cell is no longer guaranteed to match the printed table — refresh the rankings and Neuronpedia glosses accordingly.*
